# Jet Lag: The Game — Seattle Early-Game Strategy (KMZ edition)

**Goal**: Same as before — compute optimal early-game questions that efficiently bisect the possibility space.

**Key change from the original notebook**: We now load everything from `reference_data.kmz` (OSM-adapted) instead of two GTFS feeds. This dramatically simplifies the data loading while still giving us stops + route polylines.

**Architecture** (per user feedback):
- Heavy logic lives in the `jetlag/` package (plain Python, importable, testable).
- This notebook is deliberately **thin** — mostly markdown + small plain-Python exploration cells.
- Two visualization stacks:
  - **folium** → beautiful publication-quality maps (walkshed, colored routes, LayerControl).
  - **ipyleaflet** → live Python-first explorer (draw radii, click stops → immediate Python objects for strategy calculations).

**Scope**: Seattle city limits (N/S) + all of Bellevue + Redmond + **Mercer Island**.


## Quick Start

```bash
uv sync
uv run jupyter lab
```

Then open `notebooks/seattle_strategy_kmz.ipynb`.

The first time you run the loader it will fetch city boundaries via Nominatim (cached by osmnx afterwards).


In [1]:
# Robust environment setup for Jupyter (recommended when working in notebooks/)
import folium
import sys
import os
from pathlib import Path

# Detect project root reliably
if Path.cwd().name == "notebooks":
    project_root = Path.cwd().parent
else:
    project_root = Path.cwd()

# 1. Make the jetlag package importable
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 2. Change working directory so relative paths (like reference_data.kmz) work
os.chdir(project_root)

print(f"Working directory set to: {Path.cwd()}")
print("jetlag package should now import cleanly.")

# Import the functions we need (this is what was missing)
from jetlag.data import load_reference_kmz, filter_to_scope, compute_walkshed
from jetlag.viz import create_folium_map, add_attractions_to_folium

print("✅ All imports successful. You can now use load_reference_kmz(), etc.")
# Also needed for the Combined View section
from jetlag.attractions import load_seattle_attractions, CATEGORY_COLORS


Working directory set to: /Users/james/jetlag
jetlag package should now import cleanly.
✅ All imports successful. You can now use load_reference_kmz(), etc.


## 1. Load from KMZ + apply scope (4 cities incl. Mercer Island)

In [2]:
# This replaces the entire old GTFS + partridge + dedup pipeline
stops, routes = load_reference_kmz()

print("Raw stops from KMZ:", len(stops))
print("Raw route segments:", len(routes))

# Apply the exact same 4-city scope as the original notebook (now including Mercer Island)
scoped = filter_to_scope(stops)
print("\nAfter 4-city scope (Seattle + Bellevue + Redmond + Mercer Island):", len(scoped))

# Quick breakdown
print(scoped.groupby(["mode", "route"]).size().head(20))

Loaded from KMZ: 701 stops, 18 route segments
Raw stops from KMZ: 701
Raw route segments: 18
Fetching city boundary polygons via osmnx/Nominatim (4 cities incl. Mercer Island)...
  Union of 4 city polygons ready.

After 4-city scope (Seattle + Bellevue + Redmond + Mercer Island): 590
mode       route    
Link       1 Line        32
           2 Line        24
Monorail   Monorail       2
RapidRide  E             16
           RapidRide    516
dtype: int64


## 2. Compute ¼-mile walkshed

In [3]:
ws = compute_walkshed(scoped)
print("Walkshed geometry type:", type(ws).__name__)
print("Walkshed area (approx km²):", round(ws.area * 111**2, 1))  # rough degrees→km

Walkshed geometry type: MultiPolygon
Walkshed area (approx km²): 53.3


## 3. Publication-quality folium map (transit + attractions)

One clean, shareable map with stops, routes, walkshed, **and** attraction categories.
All layers are toggleable in the LayerControl.


In [4]:
# Load attractions (uses the same standing database as the dedicated notebook)
attractions = load_seattle_attractions()

m = create_folium_map(scoped, routes=routes, walkshed=ws)

# Add attraction layers (they will appear in the LayerControl)
add_attractions_to_folium(m, attractions, colors=CATEGORY_COLORS)

# Ensure LayerControl includes everything
folium.LayerControl(collapsed=False).add_to(m)

m   # Beautiful combined map with toggleable layers for transit and attractions


## 4. Ported features checklist (from original notebook)

- 4-city scope including **Mercer Island** ✓
- ¼-mile walkshed union (shaded, toggleable) ✓
- Route segment polylines (colored by RapidRide letter / Link 1 vs 2) ✓
- Color coding + LayerControl ✓
- Rich popups with route info ✓
- SAM start marker ✓
- Clean separation: data/viz logic in `jetlag/` package, notebook stays thin ✓

**New in this version**:
- Attractions from the standing database overlaid on the same map ✓
- All attraction categories toggleable in LayerControl ✓


## Next steps / Roadmap (same as original, now easier)

1. Landmark audit — add a small table of important POIs + radii.
2. Interactive question builder that uses drawn radii or selected stops from the ipyleaflet map.
3. Information-gain calculations over the scoped candidates (now trivial because everything is in GeoDataFrames).
4. Export beautiful folium maps for sharing the current best opening strategy.
